# **Tracking Refinement**

This notebook is designed to analyze the output of cell tracking and intensity measurements using a radius circle mass approach.

### **Import Libraries**
We begin by importing necessary packages, including:
- `pickle` for saving analysis results
- `tifffile` for reading/writing TIFF stacks
- `napari` for image and mask manipulation
- `cct_utils`, which contains our custom tracking logic

In [ ]:
import os
import tifffile
import numpy as np
import pickle
import napari
import matplotlib.pyplot as plt
from tqdm import tqdm
from tifffile import imwrite
cct_utils = __import__('0_cct_utils')

### **Define Paths**

Here, we:
- Set up the paths to essential data directories, including raw images, masks, and output pickle files.
- Define the base filename (`file_name`) representing the dataset to be processed.
- Construct full paths for raw images (`raw_file`), tracked masks (`mask_file`), and output pickle files (`pkl_file`) dynamically.
- This setup ensures consistency and easy switching between datasets during analysis.

In [ ]:
parent_dir = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
model_folder = "ModelAB1"
file_name = "CON_cluster1"

raw_path = os.path.join(parent_dir, "raw_data")
mask_path = os.path.join(parent_dir, "masks_tracked", model_folder)
pkl_path = os.path.join(parent_dir, "pkl_data", model_folder)
corr_path = os.path.join(parent_dir, "correlation_masks", model_folder)
seg_path = os.path.normpath(os.path.join(parent_dir, "plots", "segmentation_plots"))

raw_file = os.path.join(raw_path, f"{file_name}.tif")
mask_file = os.path.join(mask_path, f"{file_name}_mask.tif")
pkl_file = os.path.join(pkl_path, f"{file_name}.pkl")
corr_file = os.path.join(corr_path, f"{file_name}_corr.tif")

os.makedirs(seg_path, exist_ok=True)

### **Step 1: Initialize Napari Viewer and Load Raw Image and Mask**

In this step, we:
- Initialize the Napari viewer.
- Load the raw microscopy image and its corresponding mask (segmentation).
- Add the raw image and mask as layers in the viewer for visualization.

In [ ]:
# Initialize the Napari viewer
viewer = napari.Viewer()

# Load data
X = tifffile.imread(raw_file)
Y = tifffile.imread(mask_file)
print(f"Loaded raw image {X.shape} and mask {Y.shape}")

viewer.add_image(X, name="Raw")
viewer.add_labels(Y, name="Mask", opacity=0.35)

seg_filename = os.path.join(seg_path, f"{file_name}_seg_model.pdf")
cct_utils.save_napari_export(viewer, seg_filename)

### **Step 2: Filter Mask to Include Only Common Cells**

In this step, we:
- Hide the original mask layer to declutter the view.
- Identify "common cells", cells present in at least a certain percentage (80%) of frames.
- Create a filtered version of the mask containing only these common cells.
- Add the filtered mask as a new layer in the viewer for focused analysis.

In [ ]:
# Filter mask by common cells
viewer.layers['Mask'].visible = False
occurrence_limit = 80
common_cells, _ = cct_utils.get_common_cells(Y, occurrence=occurrence_limit)
print(f"Common cells: {len(common_cells)}")

Y_filtered = Y.copy()
Y_filtered[~np.isin(Y_filtered, common_cells)] = 0
viewer.add_labels(Y_filtered, name="Filtered Mask", opacity=0.35)

seg_filename = os.path.join(seg_path, f"{file_name}_seg_COM.pdf")
cct_utils.save_napari_export(viewer, seg_filename)

### **Step 3: Initialize an Empty Points Layer for Manual Label Removal**

In this step, we:
- Initialize an empty points layer where users can manually add points to mark incorrectly placed labels in the mask.
- To use this functionality, click the "Add points" button in Napari, then click on the locations of the incorrect labels.
- This step provides flexibility to remove problematic labels in the next step.

In [ ]:
# Initialize an empty points layer for manual selection
viewer.add_points(
    np.empty((0, 3), dtype=np.float32),
    size=10,
    face_color='White',
    border_color='White',
    name='Points',
    symbol='x'
)
print("An empty points layer was added. You can manually add points if needed.")

### **Step 4: Remove Labels Based on Selected Points and Save Final Mask**

In this step, we:
- Check if any points were manually added to the points layer.
- If points were added, identify the labels present at these points and remove them from the mask, creating a new layer called `Mask with Points Removed`.
- If no points were added, continue with the `Filtered Mask` layer (`Y_filtered`) for analysis and remove the empty points layer.
- Save the final mask layer (`Y_analyze`) as a `.tif` file for later use in downstream correlation analysis.
- This approach ensures that either the mask is refined manually or the default filtered mask is used, and the exact mask used for analysis is saved to disk for reproducibility.

In [ ]:
# Access the points layer data
points_data = viewer.layers['Points'].data.astype(np.uint16)

if len(points_data) > 0:
    labels_at_points = Y_filtered[points_data[:, 0], points_data[:, 1], points_data[:, 2]]
    Y_analyze = np.where(np.isin(Y_filtered, labels_at_points), 0, Y_filtered)
    viewer.add_labels(Y_analyze, name="Mask with Points Removed", opacity=0.35)
    print("Points were specified. Analyzing mask with selected points removed.")
    viewer.layers['Filtered Mask'].visible = False
    viewer.layers['Points'].visible = False
else:
    viewer.layers.remove("Points")
    Y_analyze = Y_filtered
    print("No points specified. Analyzing filtered mask.")

# Save Y_analyze to a .tif file
imwrite(corr_file, Y_analyze.astype(np.uint16))
print(f"Final mask layer (Y_analyze) saved to {corr_file}")

seg_filename = os.path.join(seg_path, f"{file_name}_seg_user.pdf")
cct_utils.save_napari_export(viewer, seg_filename)

### **Step 5: Plot Average Intensity over Time**

In this step, we:
- Calculate the average intensity for each time frame by averaging pixel intensities across the spatial dimensions (X and Y).
- Plot the average intensity trace to observe temporal intensity changes across the entire field of view.
- This plot helps identify potential trends or issues (e.g., bleaching, sudden signal drops) that may influence downstream analysis.

In [ ]:
# Plot average intensity over time
avg_intensity = np.mean(X, axis=(1, 2))
plt.figure(figsize=(15, 3))
plt.plot(avg_intensity, label="Average Intensity")
plt.legend()
plt.title("Average Intensity over Time")
plt.show()

### **Step 6: Analyze Cells for Intensities and Centers of Mass**

In this step, we:
- Set up the function to compute intensity traces and center of mass (COM) data for each cell over time.
- Use both standard and circular regions of interest (ROIs) around the COM for more precise measurements.
- Generate a new layer in Napari showing all calculated COM masks for visualization.

In [ ]:
# Analyze cells for intensities and COM
def analyze_cells(X, Y, occurrence_limit=80, radius=10):
    d = {}
    com_mask_all = np.zeros(Y.shape, dtype=np.uint8)
    for c in tqdm(np.unique(Y)[1:], desc="Processing Cells"):
        com_mask, intensities_circle, com_coords = cct_utils.get_cell_intensities_circle(c, Y, X, radius)
        intensities = cct_utils.get_cell_intensities(c, Y, X)
        d[c] = {
            'intensities': intensities,
            'intensities_circle': intensities_circle,
            'com_coords': com_coords,
            'occurrence_limit': occurrence_limit,
            'circle_radius': radius
        }
        com_mask_all += com_mask
    viewer.add_labels(com_mask_all, name="All COM Masks")
    return d

### **Step 7: Run Analysis, Check for NaNs and Save Pickle File**

In this step, we:
- Store analysis results in a dictionary (`d`) for each cell, including intensities, COM coordinates, and measurement settings.
- Check each cell's intensity data for NaN values to ensure data quality.
- Print warnings if any NaNs are found, indicating potential issues with missing or erroneous data.
- Save the analysis results to a pickle file for further processing or future reference.

In [ ]:
d = analyze_cells(X, Y_analyze, occurrence_limit=occurrence_limit)

# Check for NaNs
for cell, data in d.items():
    for key in ['intensities', 'intensities_circle']:
        if np.any(np.isnan(data[key])):
            print(f"Warning: NaNs found in {key} for cell {cell}")

# Save analysis
os.makedirs(pkl_path, exist_ok=True)
with open(pkl_file, 'wb') as f:
    pickle.dump(d, f)
print(f"Data saved to {pkl_file}")